## Step 6 — Relationship between poster clusters and fear categories

Now I want to test whether image-based clusters align with text-based fear categories.

**Inputs**
- Fear categories: `Fear_Category` from `horror_categorized_FULLSAMPLE_clean.csv`
- Cluster assignments: `embeddings_with_clusters.json` from the KMeans experiment I selected

**Process**
- Merge cluster labels into the main dataframe
- Filter to curated clusters
- For each cluster, list the top 3 fear categories with proportions

In [2]:
import json
import pandas as pd

# now we want to analyze if there is any correlation between poster clusters and fear categories
# fear categories are in column Fear_Category in the horror_categorized_FULLSAMPLE_clean.csv
# (the fear labels in this csv has been updated after re-running the LLM classification process in Google Colab)

# define paths to input files
CSV_PATH = "horror_data/horror_categorized_FULLSAMPLE_clean.csv"
SELECTED_CLUSTERS_PATH = "artifacts_posters/experiments/selected_clusters.json"
CLUSTER_ASSIGNMENTS_PATH = "artifacts_posters/experiments/kmeans_k50_pca280/embeddings_with_clusters.json"

# load the main horror movie dataset (and checking total entries)
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} movies from CSV")

# load the manually curated list of clusters and their labels
with open(SELECTED_CLUSTERS_PATH, 'r') as f:
    selected_data = json.load(f)
    selected_cluster_ids = [c['cluster_id'] for c in selected_data['clusters']]
    cluster_labels = {c['cluster_id']: c['label'] for c in selected_data['clusters']}

print(f"Selected {len(selected_cluster_ids)} clusters")

# load the cluster assignments from KMeans: maps each movie's ID to its cluster number
with open(CLUSTER_ASSIGNMENTS_PATH, 'r') as f:
    cluster_assignments = json.load(f)

# create a lookup dictionary: movie_id -> cluster_id
cluster_map = {int(item['movie_id']): item['cluster'] for item in cluster_assignments}
print(f"Cluster assignments loaded for {len(cluster_map)} movies with posters")

# add a new column to df indicating which cluster each movie belongs to
# (movies without posters will have NaN for cluster)
df['cluster'] = df['id'].map(cluster_map)

# keep only movies that are in our selected clusters (filter out the rest)
df_selected = df[df['cluster'].isin(selected_cluster_ids)].copy()
print(f"Movies in selected clusters: {len(df_selected)}")

# add cluster labels to each movie
df_selected['cluster_label'] = df_selected['cluster'].map(cluster_labels)

print("\nCluster IDs:", selected_cluster_ids)
print("Total movies in analysis:", len(df_selected))

print("\nTop 3 fear categories per cluster:")

# loop through each selected cluster and show distribution of fear categories
for cluster_id in sorted(selected_cluster_ids):
    label = cluster_labels[cluster_id]
    
    # get all movies belonging to this cluster
    cluster_movies = df_selected[df_selected['cluster'] == cluster_id]
    total = len(cluster_movies)
    
    # get the top 3 most common fear categories in this cluster
    top_3 = cluster_movies['Fear_Category'].value_counts().head(3)
    
    # print cluster header with name and ID
    print(f"\n{label} (Cluster {cluster_id})")
    
    # print each fear category with its count and percentage
    for category, count in top_3.items():
        pct = (count / total) * 100
        print(f"  {category}: {count} ({pct:.1f}%)")

Loaded 5000 movies from CSV
Selected 18 clusters
Cluster assignments loaded for 4928 movies with posters
Movies in selected clusters: 1628

Cluster IDs: [1, 3, 8, 9, 14, 16, 22, 25, 28, 30, 31, 39, 41, 42, 43, 46, 47, 48]
Total movies in analysis: 1628

Top 3 fear categories per cluster:

Shark & Aquatic Monster Horror (Cluster 1)
  Ecological / Natural Menace: 54 (74.0%)
  Invasion & Paranoia: 9 (12.3%)
  Body Horror: 5 (6.8%)

Modern Creature Feature Horror (Cluster 3)
  Invasion & Paranoia: 46 (40.4%)
  Grief & Familial Trauma: 13 (11.4%)
  Body Horror: 12 (10.5%)

Vintage Camp & Creature Schlock (1960s-80s) (Cluster 8)
  Invasion & Paranoia: 53 (34.4%)
  Captivity & Voyeuristic Sadism: 32 (20.8%)
  Possession & Loss of Agency: 27 (17.5%)

Modern Isolation & Survival Horror (Cluster 9)
  Invasion & Paranoia: 36 (35.3%)
  Grief & Familial Trauma: 14 (13.7%)
  Captivity & Voyeuristic Sadism: 12 (11.8%)

Classic Monster Movies (1950s-1960s) (Cluster 14)
  Invasion & Paranoia: 48 (46.6%

## Step 7 — Relationship between poster clusters and decades (time as a driver of aesthetics)

**Method**
The process is the same as in the previous step, but this time applied to each decade.
If a cluster dominates a decade, that suggests the cluster may be capturing an era-specific aesthetic.

In [3]:
# now we want to analyze if there is any correlation between poster clusters and decades
# decade info is in column 'decade' in the horror_categorized_FULLSAMPLE_clean.csv
# to avoid over-representation of recent decades, we will approach the analysis by decade and not by poster cluster

print("\nTop clusters per decade:")

# get unique decades from the selected data
decades = sorted(df_selected['decade'].unique())

# loop through each decade and show distribution of clusters
for decade in decades:
    # get all movies from this decade
    decade_movies = df_selected[df_selected['decade'] == decade]
    total = len(decade_movies)
    
    # get the top 5 most common clusters in this decade
    top_5 = decade_movies['cluster'].value_counts().head(5)
    
    # print decade header
    print(f"\n{decade}s (Total movies: {total})")
    
    # print each cluster with its count and percentage
    for cluster_id, count in top_5.items():
        pct = (count / total) * 100
        cluster_label = cluster_labels.get(cluster_id, "Unknown")
        print(f"  Cluster {cluster_id} ({cluster_label}): {count} ({pct:.1f}%)")


Top clusters per decade:

1950s (Total movies: 162)
  Cluster 14.0 (Classic Monster Movies (1950s-1960s)): 61 (37.7%)
  Cluster 16.0 (Japanese Vintage Horror): 43 (26.5%)
  Cluster 41.0 (Vintage Gothic (mix)): 22 (13.6%)
  Cluster 31.0 (Latin American Horror (Vintage)): 17 (10.5%)
  Cluster 46.0 (B-Movie Creature Features & Kaiju): 6 (3.7%)

1960s (Total movies: 140)
  Cluster 41.0 (Vintage Gothic (mix)): 43 (30.7%)
  Cluster 16.0 (Japanese Vintage Horror): 28 (20.0%)
  Cluster 14.0 (Classic Monster Movies (1950s-1960s)): 24 (17.1%)
  Cluster 31.0 (Latin American Horror (Vintage)): 16 (11.4%)
  Cluster 28.0 (Vintage Illustrated Monster Movies (Pre-1980)): 11 (7.9%)

1970s (Total movies: 156)
  Cluster 41.0 (Vintage Gothic (mix)): 50 (32.1%)
  Cluster 8.0 (Vintage Camp & Creature Schlock (1960s-80s)): 25 (16.0%)
  Cluster 28.0 (Vintage Illustrated Monster Movies (Pre-1980)): 20 (12.8%)
  Cluster 16.0 (Japanese Vintage Horror): 12 (7.7%)
  Cluster 14.0 (Classic Monster Movies (1950s-196

### Takeaway

Some clusters heavily dominate earlier decades (for example, classic monster imagery), while others dominate recent decades (modern ensemble horror-comedy, neon/synth palettes, etc.)

This suggests that poster clusters are capturing aesthetic styles that evolve over time, and not just subgenres or themes.

## Step 8 — Interpreting clusters with text using TF-IDF

Visual clusters can be hard to describe only by looking at their galleries, so I will also analyze the language associated with posters in each cluster using the overview texts.

In [4]:
# TF-IDF ANALYSIS FOR OVERVIEWS

import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# parsing functions

def clean_text(text):
    """Lowercase → keep letters/spaces → collapse whitespace."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[^a-zA-Z\s]", " ", text.lower())
    return re.sub(r"\s+", " ", text).strip()


# add cleaned text columns to the df
df_selected["overview_clean"] = df_selected["overview"].apply(clean_text)

# TF-IDF function to get top unigrams

def get_top_unigrams(texts, top_n=15, min_doc_count=10):
    """
    Fit TF-IDF on a list of docs (one cluster) and return unigrams
    appearing in ≥ min_doc_count documents, sorted by total TF-IDF."""
    
    # keep valid docs
    texts = [t for t in texts if isinstance(t, str) and t.strip()]
    if len(texts) < min_doc_count:
        return []

    # build TF-IDF matrix
    vec = TfidfVectorizer(stop_words="english", ngram_range=(1,1), min_df=1)
    X = vec.fit_transform(texts)

    terms = vec.get_feature_names_out()
    tfidf_sum = np.asarray(X.sum(axis=0)).ravel()
    doc_freq  = np.asarray((X > 0).sum(axis=0)).ravel()

    # filter terms by document count
    mask = doc_freq >= min_doc_count
    if not mask.any():
        return []

    idx = np.where(mask)[0]

    # sort by TF-IDF sum, take top_n
    top = idx[np.argsort(tfidf_sum[idx])[-top_n:]][::-1]

    return [(terms[i], float(tfidf_sum[i]), int(doc_freq[i])) for i in top]


In [5]:
# OVERVIEW TEXT — UNIGRAMS (min_doc_count = 8 so that we have enough data for smaller clusters)

print("\n" + "-"*80)
print("PART 1 — OVERVIEW TEXT (UNIGRAMS, doc_count ≥ 8)")
print("-"*80)

overview_unigrams = {}   # store results per cluster

for cid in sorted(selected_cluster_ids):

    # all cleaned overviews for this cluster
    texts = df_selected[df_selected["cluster"] == cid]["overview_clean"].tolist()
    label = cluster_labels.get(cid, "Unknown")

    # run TF-IDF (term must appear in ≥ 8 docs in this cluster)
    unigrams = get_top_unigrams(
        texts,
        top_n=15,
        min_doc_count=8
    )

    overview_unigrams[cid] = unigrams

    print(f"\nCluster {cid}: {label}")
    if not unigrams:
        print("  (no terms with doc_count ≥ 8)")
        continue

    print("  Top Overview Unigrams:")
    for term, score, count in unigrams:
        print(f"    {term:30s} TF-IDF sum: {score:.4f}  Doc count: {count}")


--------------------------------------------------------------------------------
PART 1 — OVERVIEW TEXT (UNIGRAMS, doc_count ≥ 8)
--------------------------------------------------------------------------------

Cluster 1: Shark & Aquatic Monster Horror
  Top Overview Unigrams:
    shark                          TF-IDF sum: 3.3789  Doc count: 24
    sharks                         TF-IDF sum: 2.6624  Doc count: 18
    fight                          TF-IDF sum: 1.7610  Doc count: 12
    white                          TF-IDF sum: 1.7441  Doc count: 13
    great                          TF-IDF sum: 1.7441  Doc count: 13
    friends                        TF-IDF sum: 1.6794  Doc count: 11
    man                            TF-IDF sum: 1.4884  Doc count: 10
    way                            TF-IDF sum: 1.4715  Doc count: 10
    deadly                         TF-IDF sum: 1.4572  Doc count: 10
    attack                         TF-IDF sum: 1.4304  Doc count: 8
    water                      

In [7]:
# replicating the analysis for clusters outside the selected ones

df["overview_clean"] = df["overview"].apply(clean_text)

df_all = df[df["cluster"].notna()].copy()
df_all["cluster_int"] = df_all["cluster"].astype(int)

print("\n" + "-"*80)
print("PART 1 — OVERVIEW TEXT (UNIGRAMS, doc_count ≥ 8)")
print("-"*80)

overview_unigrams = {}

for cid in sorted(df_all["cluster_int"].unique()):

    cluster_movies = df_all[df_all["cluster_int"] == cid]
    if len(cluster_movies) < 20:   # optional: skip tiny clusters
        continue

    texts = cluster_movies["overview_clean"].tolist()

    unigrams = get_top_unigrams(
        texts,
        top_n=15,
        min_doc_count=8
    )

    overview_unigrams[cid] = unigrams

    label = cluster_labels.get(cid, "(unlabeled cluster)")

    print(f"\nCluster {cid}: {label}  (n={len(cluster_movies)})")
    if not unigrams:
        print("  (no terms with doc_count ≥ 8)")
        continue

    for term, score, count in unigrams:
        print(f"    {term:30s} TF-IDF sum: {score:.4f}  Doc count: {count}")



--------------------------------------------------------------------------------
PART 1 — OVERVIEW TEXT (UNIGRAMS, doc_count ≥ 8)
--------------------------------------------------------------------------------

Cluster 0: (unlabeled cluster)  (n=99)
    young                          TF-IDF sum: 1.8931  Doc count: 15
    woman                          TF-IDF sum: 1.7856  Doc count: 12
    school                         TF-IDF sum: 1.6828  Doc count: 8
    revenge                        TF-IDF sum: 1.5553  Doc count: 10
    world                          TF-IDF sum: 1.5010  Doc count: 13
    life                           TF-IDF sum: 1.3623  Doc count: 10
    new                            TF-IDF sum: 1.3064  Doc count: 8
    come                           TF-IDF sum: 1.2990  Doc count: 8
    women                          TF-IDF sum: 1.2488  Doc count: 10
    group                          TF-IDF sum: 1.2444  Doc count: 9
    friends                        TF-IDF sum: 1.1876  Doc cou

### Takeaway

Running TF-IDF on **overview text** across all poster clusters revealed a clear split:

- A small subset of clusters is highly interpretable based on their vocabulary (e.g., shark/sea, vampire/castle/blood, priest/exorcism, etc).
- Many other clusters are dominated by generic horror synopsis words (e.g., young, woman, family, house, town, night, mysterious), which seem to be common across the dataset.

Although this information is valuable in itself, in order to identify more cluster-specific terms, I will rerun TF-IDF using an expanded stopword list: the default English stopwords plus the most common generic terms observed in the initial pass.

In [8]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# ---------------------------------------------------------
# expanded stopwords: dataset-specific "generic horror words"
# ---------------------------------------------------------
GENERIC_CLUSTER_WORDS = {
    "young", "woman", "women", "man", "men", "girl", "boy", "mother", "father",
    "daughter", "son", "wife", "husband", "friends", "group", "people", "family",
    "new", "old", "life", "night", "years", "soon", "begins", "finds", "discovers",
    "discover", "help", "way", "place", "secret", "secrets", "known", "past",
    "strange", "mysterious", "evil", "killer", "death", "dead", "murder", "kills",
    "house", "home", "town", "world", "film", "time"
}

# -------------------------------------
# TF-IDF helper with expanded stopwords
# -------------------------------------
def get_top_unigrams(texts, top_n=15, min_doc_count=10, extra_stopwords=None):
    """
    Fit TF-IDF on a list of docs (one cluster) and return unigrams
    appearing in ≥ min_doc_count documents, sorted by total TF-IDF.
    """
    texts = [t for t in texts if isinstance(t, str) and t.strip()]
    if len(texts) < min_doc_count:
        return []

    # --- build stopword list correctly ---
    if extra_stopwords:
        base_stopwords = TfidfVectorizer(stop_words="english").get_stop_words()
        stopwords = list(set(base_stopwords) | set(extra_stopwords))
    else:
        stopwords = "english"

    vec = TfidfVectorizer(
        stop_words=stopwords,
        ngram_range=(1, 1),
        min_df=1
    )

    X = vec.fit_transform(texts)

    terms = vec.get_feature_names_out()
    tfidf_sum = np.asarray(X.sum(axis=0)).ravel()
    doc_freq  = np.asarray((X > 0).sum(axis=0)).ravel()

    mask = doc_freq >= min_doc_count
    if not mask.any():
        return []

    idx = np.where(mask)[0]
    top = idx[np.argsort(tfidf_sum[idx])[-top_n:]][::-1]

    return [(terms[i], float(tfidf_sum[i]), int(doc_freq[i])) for i in top]



# ---------------------------------------------------------
# re-run across all clusters (overview text)
# ---------------------------------------------------------
df_all = df[df["cluster"].notna()].copy()
df_all["cluster_int"] = df_all["cluster"].astype(int)

print("\n" + "-"*80)
print("PART 1 — OVERVIEW TEXT (UNIGRAMS, doc_count ≥ 8) — EXPANDED STOPWORDS")
print("-"*80)

overview_unigrams_denoised = {}

for cid in sorted(df_all["cluster_int"].unique()):

    cluster_movies = df_all[df_all["cluster_int"] == cid]
    texts = cluster_movies["overview_clean"].tolist()

    unigrams = get_top_unigrams(
        texts,
        top_n=15,
        min_doc_count=8,
        extra_stopwords=GENERIC_CLUSTER_WORDS
    )

    overview_unigrams_denoised[cid] = unigrams

    label = cluster_labels.get(cid, "(unlabeled cluster)")
    print(f"\nCluster {cid}: {label} (n={len(cluster_movies)})")

    if not unigrams:
        print("  (no terms with doc_count ≥ 8 after expanded stopwords)")
        continue

    for term, score, count in unigrams:
        print(f"    {term:30s} TF-IDF sum: {score:.4f}  Doc count: {count}")


--------------------------------------------------------------------------------
PART 1 — OVERVIEW TEXT (UNIGRAMS, doc_count ≥ 8) — EXPANDED STOPWORDS
--------------------------------------------------------------------------------

Cluster 0: (unlabeled cluster) (n=99)
    school                         TF-IDF sum: 1.7121  Doc count: 8
    revenge                        TF-IDF sum: 1.6618  Doc count: 10
    come                           TF-IDF sum: 1.3620  Doc count: 8
    sexual                         TF-IDF sum: 1.2131  Doc count: 8
    make                           TF-IDF sum: 1.0503  Doc count: 8
    beautiful                      TF-IDF sum: 0.9876  Doc count: 8

Cluster 1: Shark & Aquatic Monster Horror (n=73)
    shark                          TF-IDF sum: 3.5409  Doc count: 24
    sharks                         TF-IDF sum: 2.7317  Doc count: 18
    white                          TF-IDF sum: 1.8287  Doc count: 13
    great                          TF-IDF sum: 1.8287  Doc cou

### Takeaway

After removing generic synopsis vocabulary and re-running TF-IDF, a small number of previously unlabeled clusters show some coherent vocabulary. For instance, clusters associated with zombie outbreaks narratives (e.g., cluster 34), mad-science or experimental horror (e.g., cluster 18), and school-centered or youth-focused settings (e.g., clusters 10). This suggests that these groups could be somewhat cohesive not only visually but also narratively.

However, many clusters continue to produce few or no stable keywords at the `doc_count ≥ 8` threshold. This reinforces an earlier observation: for several clusters, cohesion appears to be based on aesthetics and visual similarity rather than by shared narrative themes.

At the same time, overview texts are typically short and formulaic, which limits the effectiveness of TF-IDF in extracting distinctive vocabulary and may partially explain the lack of clear signals in some clusters.
